In [30]:
import sys
sys.path.insert(0, '../..')
from dependencies import *
from constants import *
from paths import *
import helper_functions

# Simple envelope TRFs
Uses the acoustic envelope as the only predictor

In [31]:
SUBJECTS = helper_functions.fuglsang_get_subjects()

In [32]:
def estimate_trfs(predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.FORWARD, padded=False, predictor_dir=FUGLSANG_PRED_CONCAT_SELF_DIR, trf_dir=FUGLSANG_TRF_SELF_DIR):

    # Ensure list of Predictors
    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]

    predictor_names = sorted(predictor_names, key=lambda p: p.value)

    model_name = helper_functions.get_trf_model_name(
        DATASET_TYPE.FUGLSANG, predictor_names, attention, model_type,GENERALISATION_TYPE.INDIVIDUAL, padded
    )

    suffix = "_padded" if padded else ""

    for subject in SUBJECTS:
        print("-" * 50)
        print(f"Processing {subject} | model: {model_name}")

        subject_trf_dir  = trf_dir / subject
        trf_path = subject_trf_dir / f"{subject}_{model_name}_trf.pickle"

        # --- Skip if exists ---
        if trf_path.exists():
            print("TRF exists, skipping.")
            continue

        # --- Load EEG ---
        eeg_path = FUGLSANG_EEG_DIR / subject / f"{subject}_eeg{suffix}.pickle"

        if not eeg_path.exists():
            print("Missing EEG, skipping.")
            continue

        eeg = eelbrain.load.unpickle(eeg_path)

        # --- Load predictors ---
        predictors = []

        for p in predictor_names:
            predictor_name = helper_functions.get_attentional_predictor_name(p, attention, padded)
            predictor_path = predictor_dir / subject / f"{predictor_name}_concat.pickle"

            if not predictor_path.exists():
                print(f"Missing predictor: {predictor_path}, skipping subject.")
                break

            predictors.append(eelbrain.load.unpickle(predictor_path))
        else:
            # --- Prepare stimulus ---
            stimulus = predictors if len(predictors) > 1 else predictors[0]

            print(f"Estimating {model_name} TRF...")

            # --- Lag handling + boosting call ---
            if model_type == MODEL_TYPE.FORWARD:
                tmin, tmax = TRF_LAG_START, TRF_LAG_END
                trf = eelbrain.boosting(
                    eeg,            # y: Response: EEG
                    stimulus,       # x: Predictor: Stimulus
                    tmin, tmax,
                    error='l1', basis=BASIS_FUNCTION_WIDTH,
                    partitions=5, test=1, selective_stopping=True
                )
            else:
                tmin, tmax = -TRF_LAG_END, -TRF_LAG_START
                trf = eelbrain.boosting(
                    stimulus,     # y: Response: Stimulus
                    eeg,          # x: Predictor: EEG
                    tmin, tmax,
                    error='l1', basis=BASIS_FUNCTION_WIDTH,
                    partitions=5, test=1, selective_stopping=True
                )

            # --- Save ---
            subject_trf_dir.mkdir(exist_ok=True, parents=True)
            eelbrain.save.pickle(trf, trf_path)

            print(f"Saved → {trf_path}")

In [33]:
trf_dirs    = [FUGLSANG_TRF_MAT_DIR,        FUGLSANG_TRF_SELF_DIR]
concat_dirs = [FUGLSANG_PRED_CONCAT_MAT_DIR, FUGLSANG_PRED_CONCAT_SELF_DIR]

for trf_dir, concat_dir in zip(trf_dirs, concat_dirs):
        print(f"Estimating TRFs for:\n- TRF_DIR: {trf_dir}\n- CONCAT_DIR: {concat_dir}\n")
        checks = [
            # Single predictors
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  False, concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  False, concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  True,  concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  True,  concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD,  False, concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD,  False, concat_dir, trf_dir),

            # Combined predictors
            ([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD, False, concat_dir, trf_dir),
            ([PREDICTOR_TYPE.ENVELOPE, PREDICTOR_TYPE.ENVELOPE_ONSET], ATTENTION_TYPE.IGNORED,  MODEL_TYPE.FORWARD, False, concat_dir, trf_dir),

            # Backward models
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False, concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False, concat_dir, trf_dir),
            #(PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, True,  CONCAT_DIR, TRF_DIR),
            #(PREDICTOR_TYPE.ENVELOPE,       ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, True,  CONCAT_DIR, TRF_DIR),
            (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, False, concat_dir, trf_dir),
            (PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.IGNORED,  MODEL_TYPE.BACKWARD, False, concat_dir, trf_dir),
        ]

        for predictors, attention, model, GENERALISATION_TYPE.INDIVIDUAL, padded, predictor_dir, trf_dir in checks:
            estimate_trfs(predictors, attention, model, GENERALISATION_TYPE.INDIVIDUAL, padded, predictor_dir, trf_dir)

        print(f"Done estimating TRFs for:\n- TRF_DIR: {trf_dir}\n- CONCAT_DIR: {concat_dir}\n")

Estimating TRFs for:
- TRF_DIR: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/TRFs/mat_file
- CONCAT_DIR: /Users/sylvestereley/Data/modelling-eeg-to-speech/fuglsang/predictors/concatenated/mat_file

--------------------------------------------------
Processing S1 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S2 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S3 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S4 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S5 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Processing S6 | model: forward_attended_envelope
TRF exists, skipping.
--------------------------------------------------
Proce

In [34]:
# SANITY CHECK: Check dimensions of estimated TRFs
for subject in SUBJECTS:
    print(f"\n{subject}")
    for predictors, attention, model, GENERALISATION_TYPE.INDIVIDUAL, padded, predictor_dir, trf_dir in checks:
        model_name = helper_functions.get_trf_model_name(DATASET_TYPE.FUGLSANG, predictors, attention, model, GENERALISATION_TYPE.INDIVIDUAL, padded)
        trf_path = trf_dir / subject / f"{subject}_{model_name}_trf.pickle"
        if trf_path.exists():
            trf = eelbrain.load.unpickle(trf_path)
            print(f"  ✓ {model_name}: {trf.h}")
        else:
            print(f"  ✗ {model_name}: MISSING")


S1
  ✓ forward_attended_envelope: <NDVar 'attended_envelope': 64 sensor, 72 time>
  ✓ forward_ignored_envelope: <NDVar 'ignored_envelope': 64 sensor, 72 time>
  ✓ forward_attended_envelope_padded: <NDVar 'attended_envelope_padded': 64 sensor, 72 time>
  ✓ forward_ignored_envelope_padded: <NDVar 'ignored_envelope_padded': 64 sensor, 72 time>
  ✓ forward_attended_envelope_onset: <NDVar 'attended_envelope_onset': 64 sensor, 72 time>
  ✓ forward_ignored_envelope_onset: <NDVar 'ignored_envelope_onset': 64 sensor, 72 time>
  ✓ forward_attended_envelope+envelope_onset: (<NDVar 'attended_envelope': 64 sensor, 72 time>, <NDVar 'attended_envelope_onset': 64 sensor, 72 time>)
  ✓ forward_ignored_envelope+envelope_onset: (<NDVar 'ignored_envelope': 64 sensor, 72 time>, <NDVar 'ignored_envelope_onset': 64 sensor, 72 time>)
  ✓ backward_attended_envelope: <NDVar 'S1_eeg': 64 sensor, 73 time>
  ✓ backward_ignored_envelope: <NDVar 'S1_eeg': 64 sensor, 73 time>
  ✓ backward_attended_envelope_onset: <N

In [35]:
# %%
# Sanity check backward TRFs
subject    = SUBJECTS[0]
model_name = helper_functions.get_trf_model_name(DATASET_TYPE.FUGLSANG, PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.BACKWARD, GENERALISATION_TYPE.INDIVIDUAL)
trf_path   = FUGLSANG_TRF_MAT_DIR / subject / f"{subject}_{model_name}_trf.pickle"

trf = eelbrain.load.unpickle(trf_path)

print("--- TRF object ---")
print(trf)
print()
print("--- h ---")
print(type(trf.h))
print(trf.h)
print()
print("--- h_scaled ---")
print(type(trf.h_scaled))
print(trf.h_scaled)
print()
print("--- r ---")
print(trf.r)
print()
print("--- proportion_explained ---")
print(trf.proportion_explained)
print()

# Check that y and x are what we expect
print("--- Training info ---")
print(f"  trf.y:    {trf.y if hasattr(trf, 'y') else 'N/A'}")
print(f"  trf.x:    {trf.x if hasattr(trf, 'x') else 'N/A'}")
print(f"  trf.tstart: {trf.tstart}")
print(f"  trf.tstop: {trf.tstop}")

--- TRF object ---
<BoostingResult attended_envelope ~ None, -1 - 0.1, error=l1, basis=0.05, partitions=5, test=1, selective_stopping=1>

--- h ---
<class 'eelbrain._data_obj.NDVar'>
<NDVar: 64 sensor, 73 time>

--- h_scaled ---
<class 'eelbrain._data_obj.NDVar'>
<NDVar: 64 sensor, 73 time>

--- r ---
0.0449651471063769

--- proportion_explained ---
-0.02929313990728377

--- Training info ---
  trf.y:    attended_envelope
  trf.x:    None
  trf.tstart: -1.0
  trf.tstop: 0.1


In [36]:
def delete_trfs(predictor_names, attention: ATTENTION_TYPE, model_type: MODEL_TYPE = MODEL_TYPE.FORWARD, padded=False, trf_dir=FUGLSANG_TRF_SELF_DIR):
    """
    Delete TRF files across all subjects so they can be recomputed.

    Args:
        predictor_names: A single PREDICTOR_TYPE or list of PREDICTOR_TYPE values.
        attention:       ATTENTION_TYPE.ATTENDED or ATTENTION_TYPE.IGNORED.
        model_type:      MODEL_TYPE.FORWARD or MODEL_TYPE.BACKWARD.
        padded:          Whether the TRF was estimated with padded data.
        trf_dir:         Directory where TRFs are stored.
    """
    if isinstance(predictor_names, PREDICTOR_TYPE):
        predictor_names = [predictor_names]

    model_name = helper_functions.get_trf_model_name(DATASET_TYPE.FUGLSANG, predictor_names, attention, model_type, GENERALISATION_TYPE.INDIVIDUAL, padded)

    deleted = 0
    missing = 0

    for subject in SUBJECTS:
        trf_path = trf_dir / subject / f"{subject}_{model_name}_trf.pickle"
        if trf_path.exists():
            trf_path.unlink()
            print(f"  ✓ Deleted {trf_path}")
            deleted += 1
        else:
            print(f"  - Not found: {trf_path}")
            missing += 1

    print(f"\nDone — {deleted} deleted, {missing} not found.")

In [37]:
# CAUTION: Irreversible deletion! Use with care.
# example delete_trfs(PREDICTOR_TYPE.ENVELOPE_ONSET, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD)
#delete_trfs(PREDICTOR_TYPE.ENVELOPE, ATTENTION_TYPE.ATTENDED, MODEL_TYPE.FORWARD, padded=True)
